In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.stats.proportion import proportions_ztest, confint_proportions_2indep

# Load data 
df = pd.read_csv('../data/processed/yrbs_cleaned_and_recoded.csv')

In [2]:
# Grouping

# Split data into male and female groups
male = df[df["WhatIsYourSex"] == 1]
female = df[df["WhatIsYourSex"] == 0]

# Proportions

# Calculate smoking proportions
p_male = male["CurrentCigaretteUse"].mean()
p_female = female["CurrentCigaretteUse"].mean()

# Calculate non-smoking proportions
male_no = 1 - p_male
female_no = 1 - p_female

# Z-test

# Count number of smokers in each group
count = np.array([
    male["CurrentCigaretteUse"].sum(),
    female["CurrentCigaretteUse"].sum()
])

# Sample sizes
nobs = np.array([
    len(male),
    len(female)
])

# Perform two-proportion z-test
z_stat, p_value = proportions_ztest(count, nobs)

# Difference + CI

# Difference in proportions (Male - Female)
diff = p_male - p_female

# Standard error of difference
se = np.sqrt(
    (p_male * (1 - p_male)) / len(male) +
    (p_female * (1 - p_female)) / len(female)
)

# 95% critical value
z_crit = 1.96

# Confidence interval
ci_lower = diff - z_crit * se
ci_upper = diff + z_crit * se

# Hypothesis

# Null and alternative hypothesis
hypothesis = (
    "H0: p_male = p_female ; "
    "H1: p_male ≠ p_female"
)

# Decision

# Significance level
alpha = 0.05

# Decision rule based on p-value
decision = (
    "Reject H0"
    if p_value < alpha
    else "Fail to reject H0"
)

# Conclusion

# Conclusion based on decision
if p_value < alpha:

    conclusion = (
        "There is sufficient statistical evidence at the 0.05 significance level "
        "to conclude that the proportion of current cigarette use is different "
        "between male and female students."
    )

else:

    conclusion = (
        "There is not sufficient statistical evidence at the 0.05 significance level "
        "to conclude that the proportion of current cigarette use is different "
        "between male and female students."
    )
    
# Interpretation
if p_value < alpha:

    if diff > 0:

        interpretation = (
            "Male students have a higher proportion "
            "of current cigarette use than female students."
        )

    elif diff < 0:

        interpretation = (
            "Female students have a higher proportion "
            "of current cigarette use than male students."
        )

    else:

        interpretation = (
            "Male and female students have the same proportion "
            "of current cigarette use."
        )

else:

    interpretation = (
        "There is no statistically significant difference "
        "in the proportion of current cigarette use between "
        "male and female students."
    )

# Summary Table
summary_df = pd.DataFrame({

    "Statistic": [

        "Male Sample Size",
        "Female Sample Size",

        "Male Smoking Proportion",
        "Male Not Smoking Proportion",

        "Female Smoking Proportion",
        "Female Not Smoking Proportion",

        "Difference (Male - Female)",

        "95% CI for Difference",

        "Hypothesis Test",

        "Z-statistic",

        "P-value",

        "Decision (α = 0.05)",

        "Conclusion at α = 0.05",

        "Interpretation in Context"
    ],

    "Value": [

        len(male),
        len(female),

        p_male,
        male_no,

        p_female,
        female_no,

        diff,

        f"({ci_lower:.6f}, {ci_upper:.6f})",

        hypothesis,

        z_stat,

        p_value,

        decision,

        conclusion,

        interpretation
    ]
})

# Format function

# Format numeric values for display
def format_value(x):

    if isinstance(x, float):

        # Use scientific notation for very small values
        if 0 < abs(x) < 0.001:
            return f"{x:.2e}"

        return f"{x:.6f}"

    return x

# Display Summary Table
from IPython.display import display

display(

    summary_df.style.format(format_value)

    .set_properties(**{
        'text-align': 'left',
        'font-family': 'Arial'
    })

    .set_table_styles([
        {
            'selector': 'th',
            'props': [
                ('text-align', 'left'),
                ('font-weight', 'bold')
            ]
        }
    ])

    .hide(axis="index")
)

# Save summary table to CSV file 
summary_df.to_csv(
    "../outputs/tables/two_sample_proportion_summary_table.csv",
    index=False,
    encoding="utf-8-sig"
)

Statistic,Value
Male Sample Size,6572
Female Sample Size,6740
Male Smoking Proportion,0.215764
Male Not Smoking Proportion,0.784236
Female Smoking Proportion,0.173145
Female Not Smoking Proportion,0.826855
Difference (Male - Female),0.042618
95% CI for Difference,"(0.029183, 0.056054)"
Hypothesis Test,H0: p_male = p_female ; H1: p_male ≠ p_female
Z-statistic,6.214820
